
---

# SQL Data Engineering Challenge

## Tables Available

```sql
ecommerce_analytics.silver.clicked_items
ecommerce_analytics.silver.silver_customers
ecommerce_analytics.silver.ordered_products
ecommerce_analytics.silver.promotions
ecommerce_analytics.silver.silver_sales_orders
```

---

# Scenario

Your company wants to create a **Gold Layer Sales Mart** for reporting.

You need to prepare clean analytical datasets from the Silver layer.

---

# TASK 1 — Build Customer Purchase Summary

## Objective

Create a customer-level summary report.

## Output

| customer_id | total_orders | total_products | total_revenue | avg_order_value |
| ----------- | ------------ | -------------- | ------------- | --------------- |

---

## Rules

* Use joins only
* No CTE
* No window functions
* Revenue = quantity × unit_price
* Ignore cancelled orders
* Average order value = total revenue / total orders

---


In [0]:
select
    s.customer_id,
    count(distinct s.order_number) as total_orders,
    sum(cast(p.qty as int)) as total_products,
    sum(cast(p.price as decimal(10,2)) * cast(p.qty as int)) as total_revenue,
    total_revenue / total_orders as avg_order_value
from ecommerce_analytics.silver.silver_sales_orders s
inner join ecommerce_analytics.silver.ordered_products p
    on s.order_number = p.order_number
group by s.customer_id
order by total_revenue desc

#### Query Logic
The query uses inner join to ensure only orders that exist in both tables are included.

1. INNER JOIN `silver_sales_orders` with `ordered_products` on `order_number` and calulates-
- total_orders: COUNT DISTINCT of order_number per customer
- total_products: SUM of quantities purchased
- total_revenue: SUM of (price × quantity)
- avg_order_value: total_revenue divided by total_orders


# TASK 2 — Find Customers Who Clicked But Never Purchased

## Output

| customer_id | total_clicks |
| ----------- | ------------ |

---

## Rules

* Use:

  * LEFT JOIN
  * NULL filtering
* Count total clicks
* Only include customers with zero purchases

---


In [0]:
SELECT 
    c.customer_id,
    COUNT(*) AS total_clicks
FROM ecommerce_analytics.silver.clicked_items c
LEFT JOIN ecommerce_analytics.silver.silver_sales_orders s
    ON c.customer_id = s.customer_id
WHERE s.customer_id IS NULL
GROUP BY c.customer_id
ORDER BY total_clicks DESC

#### Query Logic
The query uses:

1. LEFT JOIN from clicked_items to silver_sales_orders on customer_id
2. NULL filtering (WHERE s.customer_id IS NULL) to find customers with clicks but no corresponding orders
3. COUNT(*) to aggregate total clicks per customer
4. GROUP BY customer_id as required


# TASK 3 — Promotion Revenue Analysis

## Objective

Evaluate promotion performance.

## Output

| promotion_id | promotion_name | total_orders | total_revenue |
| ------------ | -------------- | ------------ | ------------- |

---

## Rules

* Include promotions with no sales
* Sort by revenue descending
* Use:

  * GROUP BY
  * aggregate functions
  * LEFT JOIN

---


In [0]:
SELECT 
    promo_list.promo_id AS promotion_id,
    CONCAT('Promo_', promo_list.promo_id) AS promotion_name,
    COUNT(DISTINCT pr.order_number) AS total_orders,
    COALESCE(SUM(CAST(op.price AS DECIMAL(10,2)) * CAST(op.qty AS INT)), 0) AS total_revenue
FROM (
    SELECT DISTINCT promo_id 
    FROM ecommerce_analytics.silver.promotions
) promo_list
LEFT JOIN ecommerce_analytics.silver.promotions pr
    ON promo_list.promo_id = pr.promo_id
LEFT JOIN ecommerce_analytics.silver.ordered_products op
    ON pr.order_number = op.order_number 
    AND pr.promo_product_id = op.id
GROUP BY promo_list.promo_id
ORDER BY total_revenue DESC

#### Query Logic
The query uses a two-level LEFT JOIN strategy:

1. Inner subquery: Gets distinct promo_ids to ensure all promotions are included
2. First LEFT JOIN: Links back to the promotions table
3. Second LEFT JOIN: Joins to ordered_products to calculate revenue (matching on both order_number and product_id)
4. COALESCE: Ensures promotions with no sales show $0 revenue instead of NULL
5. COUNT DISTINCT: Counts unique orders per promotion
6. GROUP BY: Aggregates at promotion level


# TASK 4 — Detect Duplicate Orders

## Objective

Find duplicate order IDs.

## Output

| order_id | duplicate_count |
| -------- | --------------- |

---

## Rules

* Only show duplicates
* Use:

  * GROUP BY
  * HAVING

---



In [0]:
SELECT 
    order_number AS order_id,
    COUNT(*) AS duplicate_count
FROM ecommerce_analytics.silver.silver_sales_orders
GROUP BY order_number
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC

#### Query Logic
The query uses:

1. GROUP BY order_number to group rows by order ID
2. COUNT(*) to count occurrences of each order
3. HAVING COUNT(*) > 1 to filter only duplicates (orders appearing more than once)
4. ORDER BY duplicate_count DESC to show highest duplicates first


# TASK 5 — Monthly Revenue Trend

## Objective

Create monthly sales trend analysis.

## Output

| order_month | total_orders | total_revenue |
| ----------- | ------------ | ------------- |

---

## Rules

* Aggregate monthly
* Exclude refunded/cancelled orders
* Use date formatting functions

---


In [0]:
SELECT 
    DATE_FORMAT(s.order_timestamp, 'yyyy-MM') AS order_month,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(CAST(p.price AS DECIMAL(10,2)) * CAST(p.qty AS INT)) AS total_revenue
FROM ecommerce_analytics.silver.silver_sales_orders s
INNER JOIN ecommerce_analytics.silver.ordered_products p
    ON s.order_number = p.order_number
GROUP BY DATE_FORMAT(s.order_timestamp, 'yyyy-MM')
ORDER BY order_month DESC

#### Query Logic
The query uses:

1. DATE_FORMAT() to extract year-month from order_timestamp column
2. INNER JOIN between silver_sales_orders and ordered_products
3. COUNT(DISTINCT order_number) to count unique orders per month
4. SUM(price × qty) to calculate monthly revenue
5. GROUP BY the formatted month
6. ORDER BY month to show chronological trend

```Note: The "silver_sales_orders" table doesn't have a status column, so there's no way to explicitly exclude cancelled/refunded orders based on the available schema. All orders in the table are included in the analysis.```






# TASK 6 — Product Performance Analysis

## Objective

Find the best-selling products.

## Output

| product_id | total_quantity_sold | total_revenue |
| ---------- | ------------------- | ------------- |

---

## Rules

* Sort by quantity sold descending
* Top 10 products only

---


In [0]:
SELECT 
    id AS product_id,
    SUM(CAST(qty AS INT)) AS total_quantity_sold,
    SUM(CAST(price AS DECIMAL(10,2)) * CAST(qty AS INT)) AS total_revenue
FROM ecommerce_analytics.silver.ordered_products
GROUP BY product_id
ORDER BY total_quantity_sold DESC
LIMIT 10

#### Query Logic
The query:

1. Groups by product id (product_id)
2. Calculates SUM of quantities for total units sold
3. Calculates SUM of (price × qty) for total revenue
4. Orders by quantity descending (best sellers first)
5. Limits to top 10 products only


# TASK 7 — Data Quality Validation

Write separate SQL queries to identify:

### A. Orders with missing customers

```sql
orders.customer_id IS NULL
```

---

### B. Negative quantities or prices

```sql
quantity < 0
price < 0
```

---

### C. Orders without products

Orders existing in:

```sql
sales_orders
```

but not in:

```sql
order_products
```

---


In [0]:
-- Task 7A: Orders with Missing Customers
SELECT 
    order_number,
    customer_id,
    order_timestamp
FROM ecommerce_analytics.silver.silver_sales_orders
WHERE customer_id IS NULL

In [0]:
-- Task 7B: Negative Quantities or Prices
SELECT 
    order_number,
    id AS product_id,
    product_name,
    CAST(qty AS INT) AS quantity,
    CAST(price AS DECIMAL(10,2)) AS price
FROM ecommerce_analytics.silver.ordered_products
WHERE CAST(qty AS INT) < 0 
   OR CAST(price AS DECIMAL(10,2)) < 0

In [0]:
-- Task 7C: Orders Without Products
SELECT 
    s.order_number,
    s.customer_id,
    s.customer_name,
    s.order_timestamp
FROM ecommerce_analytics.silver.silver_sales_orders s
LEFT JOIN ecommerce_analytics.silver.ordered_products p
    ON s.order_number = p.order_number
WHERE p.order_number IS NULL

#### Query Logic
- Query 7A: Simple NULL check on customer_id
- Query 7B: Filters for negative values after casting qty and price to numeric types
- Query 7C: LEFT JOIN pattern to find orders in sales_orders that don't exist in ordered_products


# BONUS TASK — Gold Layer Table Creation

Create:

```sql
gold.sales_customer_summary
```

with:

| customer_id |
| customer_name |
| total_orders |
| total_revenue |
| last_order_date |
| customer_category |

---

## Category Logic

| Category | Condition       |
| -------- | --------------- |
| VIP      | Revenue > 10000 |
| Premium  | Revenue > 5000  |
| Standard | Revenue > 1000  |
| Basic    | Else            |

---

# What This Tests

| Skill                           | Tested |
| ------------------------------- | ------ |
| JOIN Logic                      | High   |
| Aggregation                     | High   |
| Business Understanding          | High   |
| Data Validation                 | Medium |
| ETL Thinking                    | High   |
| SQL Fundamentals                | High   |
| Medallion Architecture Thinking | Medium |

---


In [0]:
-- Create gold schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS ecommerce_analytics.gold;

-- Create the gold layer table
CREATE OR REPLACE TABLE ecommerce_analytics.gold.sales_customer_summary AS
SELECT 
    s.customer_id,
    s.customer_name,
    COUNT(DISTINCT s.order_number) AS total_orders,
    SUM(CAST(p.price AS DECIMAL(10,2)) * CAST(p.qty AS INT)) AS total_revenue,
    MAX(s.order_timestamp) AS last_order_date,
    CASE 
        WHEN SUM(CAST(p.price AS DECIMAL(10,2)) * CAST(p.qty AS INT)) > 10000 THEN 'VIP'
        WHEN SUM(CAST(p.price AS DECIMAL(10,2)) * CAST(p.qty AS INT)) > 5000 THEN 'Premium'
        WHEN SUM(CAST(p.price AS DECIMAL(10,2)) * CAST(p.qty AS INT)) > 1000 THEN 'Standard'
        ELSE 'Basic'
    END AS customer_category
FROM ecommerce_analytics.silver.silver_sales_orders s
INNER JOIN ecommerce_analytics.silver.ordered_products p
    ON s.order_number = p.order_number
GROUP BY s.customer_id, s.customer_name
ORDER BY total_revenue DESC

#### Query Logic
The Gold layer table was created using:

1. INNER JOIN between silver_sales_orders and ordered_products
2. Aggregations: COUNT DISTINCT orders, SUM revenue, MAX order date
3. CASE WHEN logic for customer categorization based on revenue thresholds
4. GROUP BY customer_id and customer_name
5. ORDER BY total revenue descending

In [0]:
-- Verify the gold layer table with sample data
SELECT 
    customer_id,
    customer_name,
    total_orders,
    total_revenue,
    last_order_date,
    customer_category
FROM ecommerce_analytics.gold.sales_customer_summary
WHERE customer_category = 'Basic'
ORDER BY total_revenue DESC
LIMIT 20

In [0]:
-- Customer Category Distribution
SELECT 
    customer_category,
    COUNT(*) AS customer_count,
    SUM(total_revenue) AS category_revenue,
    AVG(total_revenue) AS avg_revenue_per_customer,
    MIN(total_revenue) AS min_revenue,
    MAX(total_revenue) AS max_revenue
FROM ecommerce_analytics.gold.sales_customer_summary
GROUP BY customer_category
ORDER BY 
    CASE customer_category
    WHEN 'VIP' THEN 1      -- VIP gets priority 1 (first)
    WHEN 'Premium' THEN 2   -- Premium gets priority 2 (second)
    WHEN 'Standard' THEN 3  -- Standard gets priority 3 (third)
    WHEN 'Basic' THEN 4     -- Basic gets priority 4 (last)
END


# Extra Engineering Questions (Verbal Round)

After SQL completion, WRITE THE FOLLOWING ANSWERS AFTER SOME RESEARCH:

1. How would you incrementalize this pipeline?
2. How would you partition large tables?
3. How would you optimize joins?
4. How would you deploy this in Fabric/Databricks?
5. How would you optimize the SQL server?
